# B1 on Colab's free T4 — Qwen2.5-7B-Instruct, 4-bit

**Explainable Financial RAG.** Runs the heavy generation passes on a free GPU so your laptop does none of it.

**This costs ₹0.** Free-tier Colab, open weights from HuggingFace, no API key anywhere. If any cell ever asks for a paid key, stop — something is wrong.

### Before you start
`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

### It is safe to be disconnected
Free Colab drops sessions, usually after a few hours. Everything here lives in Google Drive and every run checkpoints after each batch of 10 questions. If you get disconnected, reconnect and **run all cells again** — completed questions are skipped and the run resumes where it stopped. Nothing is recomputed and nothing is lost.

### Roughly how long
| Step | Time on a T4 |
|---|---|
| Setup + model download (once) | ~10 min |
| Prompt ablation, 4 arms × 50 questions | ~35 min |
| FinQA, 250 questions | ~45 min |
| TAT-QA, 250 questions | ~45 min |

You do not have to do it in one sitting. Each of the last three cells is independently resumable.

## 1. Confirm we actually have a GPU

4-bit loading is CUDA-only. Failing here with a clear message beats failing 20 minutes into a run.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell."
)
name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name}  ({total_gb:.1f} GB)")
print(f"torch {torch.__version__}")
if total_gb < 14:
    print("\nWARNING: under 14GB. 7B in 4-bit may OOM -- fall back to Qwen/Qwen2.5-3B-Instruct"
          " in the config cell below.")

## 2. Mount Drive and get the repo

The project lives in **Google Drive**, not in Colab's local disk. Colab wipes local disk on disconnect; Drive does not. That single choice is what makes checkpoints, the FAISS index and results survive the disconnects free Colab is prone to.

The next cell clones from GitHub on the first run and pulls on every run after, so the Colab copy always matches your laptop. Nothing is uploaded by hand.

If the repo is private, the clone will ask for credentials — make it public, or paste a GitHub personal access token when prompted.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

GITHUB_URL = 'https://github.com/vedh-vishnu-pogakula/finrag-explain.git'

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'finrag-explain'


def run(*cmd, cwd=None):
    """Run a git command and surface its output -- a silent failure here wastes an hour."""
    done = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print((done.stdout + done.stderr).strip())
    if done.returncode != 0:
        raise SystemExit(f'FAILED: {" ".join(cmd)}')


if (REPO / '.git').exists():
    print(f'Repo already in Drive -- pulling latest.\n')
    run('git', 'pull', '--ff-only', cwd=REPO)
else:
    if REPO.exists():
        raise SystemExit(
            f'{REPO} exists but is not a git checkout. Rename or delete it in Drive, '
            'then re-run this cell.'
        )
    print(f'Cloning {GITHUB_URL} -> {REPO} (first run only)\n')
    run('git', 'clone', GITHUB_URL, str(REPO))

os.chdir(REPO)
assert (REPO / 'src' / 'generation' / 'calculator.py').exists(), (
    'Repo looks incomplete -- did the push include the new generation files?'
)
print(f'\nWorking directory: {os.getcwd()}')

## 3. Install dependencies

Colab's own CUDA build of `torch` is kept — see the comments in `requirements-colab.txt` for why the pin is not forced here. The exact versions that resolve are recorded in step 4 so any number produced on Colab can be traced back to its environment.

In [ ]:
%%capture install_log
# Everything from the pinned local environment except torch (Colab's CUDA build stays).
!grep -vE '^\s*(#|$)|^torch==' requirements.txt > /tmp/req-colab-core.txt
!pip install -q -r /tmp/req-colab-core.txt
!pip install -q -r requirements-colab.txt
!python -m spacy download en_core_web_sm

In [ ]:
# Surface only the failures -- the full pip log is in `install_log` if you need it.
problems = [ln for ln in install_log.stdout.splitlines()
            if 'ERROR' in ln or 'incompatible' in ln]
print('\n'.join(problems) if problems else 'Install clean.')

import bitsandbytes, spacy, transformers  # noqa: F401
print(f'transformers {transformers.__version__}  bitsandbytes {bitsandbytes.__version__}')

## 4. Record the environment

A result produced in an unrecorded environment is not reproducible. This writes the resolved versions next to the results, so a Colab number and a laptop number can always be told apart later.

In [ ]:
import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version

packages = ['torch', 'transformers', 'bitsandbytes', 'accelerate',
            'sentence-transformers', 'faiss-cpu', 'spacy', 'numpy']


def _version(pkg):
    """Record a missing package as missing rather than crashing the cell."""
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


env = {
    'platform': platform.platform(),
    'python': sys.version.split()[0],
    'gpu': torch.cuda.get_device_name(0),
    'packages': {p: _version(p) for p in packages},
}
out = Path('eval/results/colab_env.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## 5. Datasets

Downloaded into Drive, so this is a one-time cost. Both are public and free.

In [ ]:
if Path('data/raw/finqa/dev.json').exists() and Path('data/raw/tatqa/dev.json').exists():
    print('Datasets already in Drive -- skipping download.')
else:
    !bash scripts/download_data.sh

## 6. Point the config at the 7B model

`local_model`, `use_program`, `n_shot` and `load_in_4bit` together are the **frozen generation configuration**. Every baseline — B1 now, B2 and B3 later — must be produced under exactly this setting, or the comparison between them means nothing. Change it here and you have to re-run all three.

In [ ]:
import yaml

CONFIG = Path('configs/config.yaml')
cfg = yaml.safe_load(CONFIG.read_text())

cfg['generation']['local_model'] = 'Qwen/Qwen2.5-7B-Instruct'  # -> 3B if the T4 OOMs
cfg['generation']['load_in_4bit'] = True
cfg['generation']['use_program'] = True
cfg['generation']['n_shot'] = 3

CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg['generation'], sort_keys=False))

## 7. Smoke test — 5 questions

Downloads the weights (~5GB, once) and proves the whole path works before committing to a two-hour run. Check that `program:` is well above 0 and `unparseable_json` is near 0; those two say the program-of-thought prompt took effect.

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset finqa --split dev --limit 5 --tag _COLAB-SMOKE --fresh 2>&1 | grep -vE 'Batches:|^\s*$' | tail -20

## 8. Prompt ablation at 7B

The same four-way comparison already measured at 1.5B, repeated at 7B. This is the cell that turns "program-of-thought helped our small model" into a claim about the prompt rather than about the model — if the ordering holds at both sizes, the finding replicates.

Resumable: rerun after a disconnect and finished arms are skipped.

In [ ]:
!python eval/baselines/run_prompt_ablation.py --dataset finqa --limit 50 --model Qwen/Qwen2.5-7B-Instruct --load-in-4bit --tag-prefix 7b 2>&1 | grep -vE 'Batches:|^\s*$'

## 9. Full B1 — FinQA, 250 questions

The headline number. ~45 min; checkpoints every 10 questions.

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset finqa --split dev 2>&1 | grep -vE 'Batches:|^\s*$' | tail -25

## 10. Full B1 — TAT-QA, 250 questions

In [ ]:
!python eval/baselines/run_b1_rag.py --dataset tatqa --split dev 2>&1 | grep -vE 'Batches:|^\s*$' | tail -25

## 11. Summary

Everything below is already saved in Drive under `eval/results/`. Nothing needs downloading.

In [ ]:
import json
from pathlib import Path

print('=== B1 end-to-end (250 questions each) ===')
for ds in ['finqa', 'tatqa']:
    p = Path(f'eval/results/b1_rag_{ds}_dev.json')
    if not p.exists():
        print(f'{ds}: not run yet')
        continue
    r = json.loads(p.read_text())
    a, c = r['answers'], r['config']
    print(f"\n{ds}  [{c['generation_model']}, {c.get('quantization')}, {c['prompt_version']}]")
    print(f"  numeric_accuracy = {a['numeric_accuracy']}   span_f1 = {a['span_token_f1']}")
    print(f"  recall@5 = {r['overall'].get('recall@5')}   citation_precision = "
          f"{r['citation_precision']}")
    print(f"  program_rate = {r['program_rate']}   unparseable = {r['unparseable_json_rate']}")
    print(f"  failures = {r['failure_modes']}")

abl = Path('eval/results/ablation_finqa_dev_7b.md')
if abl.exists():
    print('\n' + abl.read_text())

## Done

Results are in Drive at `finrag-explain/eval/results/`:

| File | What it is |
|---|---|
| `b1_rag_{finqa,tatqa}_dev.json` | headline B1 numbers |
| `checkpoints/b1_rag_*.jsonl` | every question, its evidence, expression and score — this is what the failure taxonomy is built from |
| `ablation_finqa_dev_7b.{json,md}` | the 7B prompt ablation table |
| `colab_env.json` | the environment these numbers came from |

To bring them back to the laptop, copy the `eval/results/` folder out of Drive — the per-question `.jsonl` files matter as much as the summaries, since the failure analysis is done on them.

**Next:** Month 5 — the evidence-grounding layer and RAGAS (B2). Before writing any B2 code, read the RAGAS note in `CLAUDE.md`: it defaults to an OpenAI judge and bills silently on the first call, so the judge has to be set explicitly to a local or free model *before* that call, not after.